# Create FLUT

The test requires a non-reactive FLUT with variable enthalpy/temperature and constant mixture.

In [ ]:
import pyFLUT
from pyFLUT import flut_processing_utilities as fpu
import cantera as ct
import numpy as np

Load mechanism

In [ ]:
gas = ct.Solution('ch4_smooke_EKT.yaml')

In [ ]:
gas.species_names

### Create manifold

We create a manifold whose temperature is wider than the expected (particle and gas @450 K)

In [ ]:
# Air
air = {'O2': 0.233, 'N2': 0.767}

fuel = {'CH4': 1}

# Temperatures
T0s = np.arange(300,1501,100)

# Fuel
Zs = np.arange(0,1.01,0.05)

parameters = tuple( [Zs] + [T0s] )

# Generate meshgrid
combination_tuple = np.meshgrid(*parameters , indexing='ij')

# Combine into 2xn shape
combination_array = np.concatenate(tuple([x[...,np.newaxis] for x in combination_tuple]),axis=len(combination_tuple))

combination_array.shape

Create solution array

In [ ]:
sol_arr = ct.SolutionArray(gas,combination_array[...,0].shape)

In [ ]:
species_array = np.zeros(tuple(list(combination_array.shape[:-1]) + [gas.n_species]))
for spec, Y in air.items():
    species_array[...,gas.species_index(spec)] += Y * (1 - combination_array[...,0])
for spec, Y in fuel.items():
    species_array[...,gas.species_index(spec)] +=  Y * combination_array[...,0]

In [ ]:
sol_arr.TPY = combination_array[...,-1], ct.one_atm, species_array

### Build flut from the data

In [ ]:
data = np.concatenate((sol_arr.T[...,np.newaxis],sol_arr.P[...,np.newaxis],sol_arr.density[...,np.newaxis],sol_arr.enthalpy_mass[...,np.newaxis],sol_arr.Y),axis=len(combination_array.shape)-1)

In [ ]:
sol_arr.equilibrate("HP")
burnt_data = np.concatenate((sol_arr.T[...,np.newaxis],sol_arr.P[...,np.newaxis],sol_arr.density[...,np.newaxis],sol_arr.enthalpy_mass[...,np.newaxis],sol_arr.Y),axis=len(combination_array.shape)-1)
data = np.concatenate((data[...,np.newaxis,:],burnt_data[...,np.newaxis,:]),axis=-2)

In [ ]:
import collections

input_dict = {'Z':Zs}
input_dict.update({'T0':T0s})
input_dict.update({'yc_norm':np.array([0,1])})

input_dict = collections.OrderedDict(input_dict)
output_list = [(name,i) for i, name in enumerate(['T','p','rho','ha'])]
output_list = output_list + [(name, i + len(output_list)) for i, name in enumerate(gas.species_names)]
output_dict = collections.OrderedDict(output_list)

flut = pyFLUT.Flut(data,input_dict=input_dict,output_dict=output_dict)
flut.gas = gas
flut

In [ ]:
var = 'CO2'
np.max(flut[var]), np.min(flut[var])

In [ ]:
flut.calc_non_dimensional_quantity(along_variable='T0',quantity='ha')
flut.map_variables(from_inp='T0',to_inp='ha_norm',n_points={'method': 'arange', 'values': [0,1.001,0.1]})
flut.calc_progress_variable(definition_dict={"CO2":1, "H2O":1, "CO": 1, "H2": 1}, along_variable="yc_norm")
flut.map_variables(from_inp='yc_norm',to_inp='yc_norm',n_points={'method': 'arange', 'values': [0,1.001,0.1]})
flut

In [ ]:
flut.contourdata("yc","ha","T",Z =0.4)

In [ ]:
flut.input_dict

Calc additionally needed variables and rename (maybe this can be adapted in the future)

In [ ]:
flut.calc_with_cantera(['rho', 'cp', 'lambda', 'viscosity','thermal_diffusivity'],gas=gas)
flut.rename_variables(rename_dict={'viscosity':'mu'})

In [ ]:
flut.output_dict

In [ ]:
flut = fpu.calc_OpenFOAM_quantities(flut)
flut = fpu.prepare_export(flut)
flut.write_h5('FLUT.h5')

### Build BC Flut

In [ ]:
input_dict = {'Z':Zs}
input_dict.update({'T0':T0s})
input_dict.update({'yc_norm':np.array([0,1])})

input_dict = collections.OrderedDict(input_dict)
output_list = [(name,i) for i, name in enumerate(['T','p','rho','ha'])]
output_list = output_list + [(name, i + len(output_list)) for i, name in enumerate(gas.species_names)]
output_dict = collections.OrderedDict(output_list)

flut_BC = pyFLUT.Flut(data,input_dict=input_dict,output_dict=output_dict)
flut_BC.gas = gas
flut_BC

In [ ]:
flut_BC.calc_progress_variable(definition_dict={"CO2":1, "H2O":1, "CO": 1, "H2": 1}, along_variable="yc_norm")
flut_BC = fpu.copy_yc_boundaries(flut_BC,ha_norm_input_variable="T0")
flut_BC.calc_non_dimensional_quantity(along_variable='yc_norm',quantity='yc',output_suffix="_BC_norm")
flut_BC.map_variables(from_inp='yc_norm',to_inp='yc_BC_norm',n_points={'method': 'arange', 'values': [0,1.001,0.1]})
flut_BC = fpu.swap_input_variables(flut_BC, swap_keys= ["T0", "yc_BC_norm"])
flut_BC.calc_non_dimensional_quantity(along_variable='T0',quantity='T')
flut_BC.map_variables(from_inp='T0',to_inp='T_norm',n_points={'method': 'arange', 'values': [0,1.001,0.1]})
flut_BC.rename_variables(rename_dict={"yc_min": "yc_BC_min", "yc_max": "yc_BC_max"})
flut_BC

In [ ]:
flut_BC = fpu.calc_OpenFOAM_quantities(flut_BC)
flut_BC = fpu.prepare_export(flut_BC)
flut_BC

In [ ]:
flut_BC.normalization_dict["yc_BC_norm"]["Settings"]["Physical_name"] = "yc"
flut_BC.normalization_dict["T_norm"]["Settings"]["Physical_name"] = "T_BC"
flut_BC.normalization_dict

In [ ]:
flut_BC.output_variables

In [ ]:
flut_BC.contourdata("yc","T_norm","T",Z=0.3)

In [ ]:
flut_BC.write_h5("FLUT_BC.h5", normalization_dict=flut_BC.normalization_dict)

# Set up initial conditions 

Randomly choose conditions 

In [ ]:
flut_BC.input_dict["T_norm"]

In [ ]:
random_Zs = np.random.choice(flut.input_dict["Z"], 100)
random_Zs = np.linspace(flut.input_dict["Z"][0],flut.input_dict["Z"][-1],100) # from earlier
ha_norms = np.random.choice(flut.input_dict["ha_norm"], 100)
ha_norms = np.linspace(flut.input_dict["ha_norm"][0],flut.input_dict["ha_norm"][-1],100) # from earlier
yc_norms = np.random.choice(flut.input_dict["yc_norm"], 100)
yc_norms = np.linspace(flut.input_dict["yc_norm"][0],flut.input_dict["yc_norm"][-1],100)
random_T_norms = np.random.choice(flut_BC.input_dict["T_norm"], 100) # from earlier
random_T_norms = np.linspace(flut_BC.input_dict["T_norm"][0],flut_BC.input_dict["T_norm"][-1],100) # from earlier
# print("["+", ".join(f"{x:.1f}" for x in Zs)+"]")
# print("["+", ".join(f"{x:.1f}" for x in ha_norms)+"]")
# print("["+", ".join(f"{x:.1f}" for x in yc_norms)+"]")
print("["+", ".join(f"{x:.2f}" for x in random_T_norms)+"]")

random_Zs = np.array([0.6, 0.9, 0.0, 0.6, 0.9, 0.5, 0.5, 0.4, 0.0, 1.0, 0.8, 0.6, 0.4, 0.7, 0.6, 0.2, 0.7, 0.9, 0.1, 1.0, 0.1, 0.8, 0.9, 0.9, 0.0, 0.8, 0.2, 0.8, 0.4, 0.6, 0.4, 0.5, 0.8, 0.3, 0.4, 0.1, 0.6, 0.1, 0.2, 1.0, 1.0, 0.8, 1.0, 1.0, 0.6, 0.9, 0.7, 0.8, 0.7, 0.5, 0.3, 0.4, 0.2, 0.9, 0.2, 0.7, 0.6, 0.5, 0.1, 0.8, 0.3, 0.9, 0.4, 0.7, 0.2, 0.8, 0.0, 0.9, 0.4, 0.4, 0.8, 0.2, 0.2, 0.9, 1.0, 0.4, 0.2, 0.8, 0.7, 0.0, 0.9, 0.5, 0.5, 0.3, 0.2, 0.2, 0.1, 1.0, 0.4, 0.6, 0.5, 0.1, 0.5, 1.0, 0.8, 0.0, 0.0, 0.1, 0.7, 0.6])
ha_norms = np.array([0.2, 0.7, 0.6, 0.1, 0.7, 0.3, 0.5, 0.0, 0.7, 0.6, 0.5, 0.9, 0.8, 0.0, 0.9, 0.6, 0.2, 0.8, 1.0, 0.4, 0.2, 0.0, 0.0, 0.0, 0.2, 0.3, 0.5, 0.5, 0.3, 0.0, 0.4, 0.0, 0.3, 0.4, 0.6, 0.8, 1.0, 0.0, 0.4, 0.4, 0.7, 1.0, 0.8, 0.7, 0.7, 0.6, 0.2, 0.0, 0.4, 0.1, 0.8, 1.0, 0.4, 0.2, 0.3, 0.2, 0.8, 0.4, 0.1, 0.2, 0.2, 0.5, 0.8, 0.5, 0.9, 1.0, 0.5, 0.9, 0.8, 0.2, 0.5, 0.6, 0.7, 0.8, 1.0, 0.4, 0.5, 0.5, 0.2, 0.1, 1.0, 0.3, 0.5, 0.5, 0.5, 1.0, 0.2, 0.9, 0.2, 0.0, 0.0, 0.6, 0.9, 0.9, 0.0, 0.1, 0.7, 0.8, 0.7, 0.3])
yc_norms = np.array([1.0, 0.4, 0.7, 0.8, 1.0, 1.0, 1.0, 0.1, 0.7, 0.2, 1.0, 0.6, 0.5, 0.4, 0.2, 0.7, 0.4, 0.5, 0.9, 0.2, 0.2, 0.1, 0.9, 0.9, 0.9, 0.7, 0.6, 0.9, 0.8, 0.2, 0.0, 1.0, 0.4, 1.0, 0.1, 0.1, 0.8, 0.6, 0.3, 1.0, 0.9, 0.6, 0.4, 0.0, 0.0, 0.8, 0.6, 0.2, 0.4, 0.1, 0.7, 0.2, 0.2, 0.4, 0.2, 0.3, 0.7, 0.9, 0.3, 0.1, 0.3, 0.9, 0.7, 0.7, 0.2, 0.9, 0.8, 0.3, 0.6, 0.7, 0.4, 0.4, 0.2, 1.0, 0.5, 0.5, 0.1, 0.9, 0.7, 0.0, 0.6, 0.8, 0.5, 0.1, 0.9, 0.0, 0.0, 0.6, 0.4, 0.4, 0.4, 0.8, 0.4, 0.9, 0.1, 0.9, 0.6, 0.5, 0.2, 0.4])
random_T_norms = np.array([0.3, 0.4, 0.7, 1.0, 0.2, 0.8, 0.7, 0.3, 0.4, 1.0, 0.5, 0.7, 0.5, 1.0, 0.5, 0.5, 0.2, 0.7, 0.4, 0.7, 0.7, 0.1, 0.3, 0.3, 1.0, 0.1, 0.7, 0.2, 0.5, 0.3, 0.2, 0.5, 0.2, 1.0, 0.3, 0.1, 0.3, 0.5, 0.2, 0.8, 0.2, 0.5, 0.8, 0.4, 0.0, 0.3, 0.6, 0.8, 0.0, 0.8, 0.5, 0.8, 0.3, 1.0, 1.0, 0.4, 0.1, 0.4, 0.6, 0.6, 0.4, 0.1, 0.8, 0.5, 0.4, 0.3, 0.8, 0.2, 1.0, 0.2, 0.8, 0.7, 0.1, 0.2, 0.3, 1.0, 0.9, 0.0, 0.4, 0.5, 0.2, 1.0, 0.7, 0.8, 0.7, 0.3, 0.0, 0.3, 0.0, 1.0, 0.8, 0.0, 0.1, 0.3, 0.3, 1.0, 0.0, 0.2, 0.1, 1.0])

In [ ]:
points = np.concatenate((random_Zs[:,np.newaxis],ha_norms[:,np.newaxis],yc_norms[:,np.newaxis]),axis=1)
ha = flut.lookup(points,"ha")
yc = flut.lookup(points,"yc")

In [ ]:
points_BC = np.concatenate((random_Zs[:,np.newaxis],yc_norms[:,np.newaxis],random_T_norms[:,np.newaxis]),axis=1)
random_Ts = flut_BC.lookup(points_BC,"T")

In [ ]:
flut.normalization_dict

In [ ]:
import python_utilities
import pandas as pd

In [ ]:
df = pd.DataFrame({"Z": random_Zs, "ha": ha, "yc": yc})

In [ ]:
for var in list(df.columns):
    if True:

        field_info = {
        'class': 'volScalarField', 
        'location': "foam_template/0.org", 
        'object': var, 
        'dimension': '[0 0 0 0 0 0 0]',
        'values': np.array(df[var]),
        'uniformValue': None,
        'boundary': {
        'patch': ['OUTLET', 'INLET', 'WALL'],
        'type': ['zeroGradient', 'zeroGradient', 'zeroGradient'], 
        'value': [None,None,None]
        }
        }
        if var == 'ha':
            field_info['dimension'] = '[0 2 -2 0 0 0 0]'
            field_info['boundary']['type'] = ['zeroGradient', 'zeroGradient', 'boundaryLookUp']
            field_info['boundary']['value'][-1] = 0
        python_utilities.foamTools.write_foam_file(field_info)

In [ ]:
random_Ts_4x = np.repeat(random_Ts[np.newaxis,:], repeats=4, axis=0).ravel()

field_info = {
'class': 'volScalarField', 
'location': "foam_template/0.org", 
'object': "T_BC", 
'dimension': '[0 0 0 0 0 0 0]',
'values': 0,
'uniformValue': None,
'boundary': {
'patch': ['OUTLET', 'INLET', 'WALL'],
'type': ['zeroGradient', 'zeroGradient', 'fixedValue'], 
'value': [None,None,random_Ts_4x]
}
}
python_utilities.foamTools.write_foam_file(field_info)

In [ ]:
np.repeat(random_Ts[np.newaxis,:], repeats=4, axis=0).ravel().reshape(4,100)

In [ ]:
import pyvista as pv

In [ ]:
reader = pv.OpenFOAMReader("foam_template/foam.foam")
reader.set_active_time_value(0.001)
vtk = reader.read()

In [ ]:
cc = vtk['boundary']['WALL'].cell_centers().points

In [ ]:
cc[:,2].reshape(4,100)

In [ ]:
w = vtk['boundary']['WALL']

In [ ]:
np.max(np.abs(w["T"] - w["T_BC"]))

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
plt.plot(vtk["boundary"]["WALL"].cell_centers().points[:100,2], vtk["boundary"]["WALL"]["T"][:100])
plt.plot(vtk["boundary"]["WALL"].cell_centers().points[:100,2], np.abs(vtk["boundary"]["WALL"]["T"][:100] - vtk["boundary"]["WALL"]["T_BC"][:100]))
plt.plot(vtk["boundary"]["WALL"].cell_centers().points[:100,2], vtk["boundary"]["WALL"]["T_BC"][:100])